<a href="https://colab.research.google.com/github/evelynn-rhodes/Kira_Lab_AllenSDK001/blob/main/DownloadSDK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install a modern NumPy that works with Python 3.12
!pip install "numpy>=1.24,<2.0"

# 2. Install AllenSDK without trying to downgrade NumPy
!pip install allensdk --no-deps

# 3. Manually install its dependencies
!pip install psycopg2-binary hdmf h5py matplotlib pandas requests scipy scikit-learn tqdm
!pip install argschema boto3 glymur ndx-events pynrrd pynwb scikit-build semver SimpleITK

In [3]:
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

# Choose a directory for cache (Colab local in /content)
# These files would appear on when you click the folder icon to the left
boc = BrainObservatoryCache(manifest_file='/content/manifest.json')

# Get ALL experiment metadata that include static/drifting gratings
exps = boc.get_ophys_experiments(stimuli=['static_gratings'])

len(exps), exps[:5]   # show count and first few entries

#For these entries, the most important things are the experiment ID,
#the targeted struture (we made sure to grab the experiment with imaging in the primary cortex (VISp))
# The session_type is also important as it decribes if the stimuli was drifting or grating, session A has many drifitng gratings and session B has many static gratings


(456,
 [{'id': 663488086,
   'imaging_depth': 175,
   'targeted_structure': 'VISl',
   'cre_line': 'Slc17a7-IRES2-Cre',
   'reporter_line': 'Ai93(TITL-GCaMP6f)',
   'acquisition_age_days': 124,
   'experiment_container_id': 662358769,
   'session_type': 'three_session_B',
   'donor_name': '361636',
   'specimen_name': 'Slc17a7-IRES2-Cre;Camk2a-tTA;Ai93-361636',
   'fail_eye_tracking': False},
  {'id': 503772253,
   'imaging_depth': 175,
   'targeted_structure': 'VISpm',
   'cre_line': 'Cux2-CreERT2',
   'reporter_line': 'Ai93(TITL-GCaMP6f)',
   'acquisition_age_days': 103,
   'experiment_container_id': 511510822,
   'session_type': 'three_session_B',
   'donor_name': '225037',
   'specimen_name': 'Cux2-CreERT2;Camk2a-tTA;Ai93-225037',
   'fail_eye_tracking': True},
  {'id': 662960692,
   'imaging_depth': 175,
   'targeted_structure': 'VISrl',
   'cre_line': 'Cux2-CreERT2',
   'reporter_line': 'Ai93(TITL-GCaMP6f)',
   'acquisition_age_days': 128,
   'experiment_container_id': 662351162,

In [4]:
# Batch pick & export 4 Allen Brain Observatory experiments:
# - For drifiting_gratings + three_session_A
# - static_gratings + three_session_B
# - prefer diverse targeted_structure; fill if not enough unique
# - export stim table + timestamps + dF/F + per-experiment metadata + overall index

import os, sys, subprocess
import numpy as np
import pandas as pd


N_EXPTS = 4
OUT_ROOT = "/content"
STIM_NAME = "static_gratings"
SESSION_TYPE = "three_session_B"

boc = BrainObservatoryCache()

# 1) List experiments that match our criteria
exp_meta = boc.get_ophys_experiments(stimuli=[STIM_NAME], session_types=[SESSION_TYPE])
exp_df = pd.DataFrame(exp_meta)

# 2) Prefer unique areas, then fill to reach N_EXPTS
exp_df = exp_df.sort_values(by=["targeted_structure", "imaging_depth", "id"]).reset_index(drop=True)

#Just to help with picking unique areas
picked = []
seen_areas = set()

# First pass: one per area
for _, r in exp_df.iterrows():
    area = r.get("targeted_structure", None)
    if area and area not in seen_areas:
        picked.append(r)
        seen_areas.add(area)
    if len(picked) >= N_EXPTS:
        break

# Second pass: fill if needed
if len(picked) < N_EXPTS:
    picked_ids = {int(x["id"]) for x in picked}
    for _, r in exp_df.iterrows():
        if int(r["id"]) not in picked_ids:
            picked.append(r)
            picked_ids.add(int(r["id"]))
        if len(picked) >= N_EXPTS:
            break

chosen = pd.DataFrame(picked).reset_index(drop=True)

# 3) Save the selection index (only columns that actually exist)
index_cols = [c for c in [
    "id","experiment_container_id","targeted_structure","imaging_depth",
    "session_type","cre_line","reporter_line","donor_name"
] if c in chosen.columns]
index_path = os.path.join(OUT_ROOT, "selected_experiments.csv")
chosen[index_cols].to_csv(index_path, index=False)
print(f"Saved index: {index_path}")
print(chosen[index_cols])

# 4) Export helpers
def _save_df(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    print("  ->", path)

# 5) Download & export each experiment
for _, row in chosen.iterrows():
    exp_id = int(row["id"])
    exp_dir = os.path.join(OUT_ROOT, f"exp_{exp_id}")
    os.makedirs(exp_dir, exist_ok=True)
    print(f"\n=== Processing experiment {exp_id} | {row.get('targeted_structure','?')} | {row.get('session_type','?')} ===")

    # Open/download the experiment
    ds = boc.get_ophys_experiment_data(exp_id)

    # a) Stimulus table for gratings (has 'start'/'end' frame indices, orientation, spatial_frequency, etc.)
    stim_table = ds.get_stimulus_table(STIM_NAME)
    _save_df(stim_table.reset_index(drop=True), os.path.join(exp_dir, f"static_gratings_stim_table_{exp_id}.csv"))

    # b) dF/F traces & timestamps
    cell_ids = ds.get_cell_specimen_ids()
    # Brain Observatory 1.1 returns (timestamps, traces) for get_dff_traces
    timestamps, traces = ds.get_dff_traces(cell_specimen_ids=cell_ids)

    ts_df = pd.DataFrame({"time_s": np.asarray(timestamps, dtype=float)})
    _save_df(ts_df, os.path.join(exp_dir, f"timestamps_{exp_id}.csv"))

    traces = np.asarray(traces, dtype=float)
    dff_cols = [f"t{i}" for i in range(traces.shape[1])]
    dff_df = pd.DataFrame(traces, columns=dff_cols)
    dff_df.insert(0, "cell_specimen_id", cell_ids)
    _save_df(dff_df, os.path.join(exp_dir, f"dff_traces_{exp_id}.csv"))

    # c) Per-experiment metadata (only fields that exist in your listing)
    meta = {
        k: row.get(k, None) for k in [
            "id","experiment_container_id","targeted_structure","imaging_depth",
            "session_type","cre_line","reporter_line","donor_name"
        ] if k in row.index
    }
    meta_df = pd.DataFrame([meta])
    _save_df(meta_df, os.path.join(exp_dir, f"metadata_{exp_id}.csv"))

print("\nDone. Open the left Files pane to download the CSVs (or right-click the exp_* folders to zip & download).")



2026-05-22 13:31:48,106 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/514549621
INFO:allensdk.api.api.retrieve_file_over_http:Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/514549621


Saved index: /content/selected_experiments.csv
          id  experiment_container_id targeted_structure  imaging_depth  \
0  501889084                511510797              VISal            175   
1  557182484                556899621              VISam            175   
2  501800164                511510688               VISl            175   
3  500964514                511509529               VISp            175   

      session_type      cre_line       reporter_line donor_name  
0  three_session_B  Cux2-CreERT2  Ai93(TITL-GCaMP6f)     225036  
1  three_session_B  Cux2-CreERT2  Ai93(TITL-GCaMP6f)     268773  
2  three_session_B  Cux2-CreERT2  Ai93(TITL-GCaMP6f)     225039  
3  three_session_B  Cux2-CreERT2  Ai93(TITL-GCaMP6f)     222420  

=== Processing experiment 501889084 | VISal | three_session_B ===
  -> /content/exp_501889084/static_gratings_stim_table_501889084.csv
  -> /content/exp_501889084/timestamps_501889084.csv


2026-05-22 13:32:39,711 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/557509351
INFO:allensdk.api.api.retrieve_file_over_http:Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/557509351


  -> /content/exp_501889084/dff_traces_501889084.csv
  -> /content/exp_501889084/metadata_501889084.csv

=== Processing experiment 557182484 | VISam | three_session_B ===
  -> /content/exp_557182484/static_gratings_stim_table_557182484.csv
  -> /content/exp_557182484/timestamps_557182484.csv


2026-05-22 13:33:05,200 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/516802878
INFO:allensdk.api.api.retrieve_file_over_http:Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/516802878


  -> /content/exp_557182484/dff_traces_557182484.csv
  -> /content/exp_557182484/metadata_557182484.csv

=== Processing experiment 501800164 | VISl | three_session_B ===
  -> /content/exp_501800164/static_gratings_stim_table_501800164.csv
  -> /content/exp_501800164/timestamps_501800164.csv


2026-05-22 13:33:49,654 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/514408643
INFO:allensdk.api.api.retrieve_file_over_http:Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/514408643


  -> /content/exp_501800164/dff_traces_501800164.csv
  -> /content/exp_501800164/metadata_501800164.csv

=== Processing experiment 500964514 | VISp | three_session_B ===
  -> /content/exp_500964514/static_gratings_stim_table_500964514.csv
  -> /content/exp_500964514/timestamps_500964514.csv
  -> /content/exp_500964514/dff_traces_500964514.csv
  -> /content/exp_500964514/metadata_500964514.csv

Done. Open the left Files pane to download the CSVs (or right-click the exp_* folders to zip & download).
